In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("hpd_orders.csv")

df.head()

In [ ]:
channel = df.groupby("channel")["order_amount"].sum()

channel

In [ ]:
import plotly.express as px

In [ ]:
fig = px.bar(
    x = channel.index,
    y = channel.values,
    title = "Total Sales by Channel"
)

In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"

fig.show()

In [ ]:
user_question = "Which channel generated the highest total sales?"

print(user_question)

In [ ]:
if user_question == "Which channel generated the highest total sales?":
    result = (
        df.groupby("channel")["order_amount"]
        .sum()
        .sort_values(ascending=False)
    )

result

In [ ]:
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = api_key)

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Say hello to my AI marketing analytics project."
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
channel_summary = (
    df.groupby("channel")["order_amount"]
    .sum()
    .sort_values(ascending=False)
)

channel_summary

In [ ]:
summary_text = channel_summary.to_string()

print(summary_text)

In [ ]:
prompt = f"""
You are a marketing data analyst.

Here is the sales summary by channel:

{summary_text}

Please analyze:
1. Which channel performs best
2. Which channel performs worst
3. Give one business insight
4. Keep the answer concise and professional
"""

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
def generate_ai_insight(summary_text, question):
    prompt = f"""
You are a marketing data analyst.

Business question:
{question}

Data summary:
{summary_text}

Please provide:
1. Direct answer
2. Key business insight
3. One recommendation

Keep the answer concise and professional.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
user_question = "Which neighborhood generated the highest total sales?"

In [ ]:
neighborhood_summary = (
    df.groupby("neighborhood")["order_amount"]
    .sum()
    .sort_values(ascending=False)
)

summary_text = neighborhood_summary.to_string()

print(summary_text)

In [ ]:
result = generate_ai_insight(
    summary_text,
    user_question
)

print(result)

In [ ]:
def run_analysis(question):
    question = question.lower()

    if "channel" in question:
        group_col = "channel"
    elif "neighborhood" in question or "region" in question:
        group_col = "neighborhood"
    elif "category" in question or "product" in question:
        group_col = "product_category"
    elif "season" in question:
        group_col = "season"
    else:
        return "Sorry, I can only analyze channel, neighborhood, product category, or season for now."

    summary = (
        df.groupby(group_col)["order_amount"]
        .sum()
        .sort_values(ascending=False)
    )

    summary_text = summary.to_string()

    insight = generate_ai_insight(summary_text, question)

    return summary, insight

In [ ]:
summary, insight = run_analysis(
    "Which product category generated the highest total sales?"
)

summary

In [ ]:
print(insight)

In [ ]:
def run_analysis(question):

    question_lower = question.lower()

    # -------------------------
    # Step 1: detect groupby column
    # -------------------------

    if "channel" in question_lower:
        group_col = "channel"

    elif "neighborhood" in question_lower or "region" in question_lower or "area" in question_lower:
        group_col = "neighborhood"

    elif "category" in question_lower or "product" in question_lower:
        group_col = "product_category"

    elif "season" in question_lower:
        group_col = "season"

    else:
        return None, "Unsupported analysis dimension."


    # -------------------------
    # Step 2: detect metric
    # -------------------------

    if "average" in question_lower or "avg" in question_lower:

        result = (
            df.groupby(group_col)["order_amount"]
            .mean()
            .sort_values(ascending=False)
        )

    elif "count" in question_lower or "number" in question_lower or "most orders" in question_lower:

        result = (
            df.groupby(group_col)["order_id"]
            .count()
            .sort_values(ascending=False)
        )

    else:

        # default = total sales
        result = (
            df.groupby(group_col)["order_amount"]
            .sum()
            .sort_values(ascending=False)
        )


    # -------------------------
    # Step 3: AI insight
    # -------------------------

    result_text = result.to_string()

    insight = generate_ai_insight(
        result_text,
        question
    )

    return result, insight

In [ ]:
summary, insight = run_analysis(
    "Which channel generated the highest average order value?"
)

summary

In [ ]:
print(insight)